In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Model, load_model
import joblib
from xgboost import XGBRegressor

In [2]:
save = 0 #Flag to save results into file

In [3]:
pr_file_name = 'my_pr_test_optuna.csv'
cons_file_name='my_cons_test_optuna.csv'

In [4]:
#Loading the deep learning model
#saved_model = load_model('tuned_dl_model.h5', compile=False)

In [5]:
#Loading the optuna xgboost model
saved_model = XGBRegressor()
saved_model.load_model('tuned_xgboost_model.json')

In [6]:
#Loading the grid search xgboost model
#import joblib
#saved_model = joblib.load('xgb_model_grid_search.pkl')

In [7]:
data_dir = "data/"
test_features = pd.read_csv(f'{data_dir}test_processed.csv')
test_features

,hsize,num_children5,num_children10,num_children18,age,num_adult_female,num_adult_male,num_elderly,sworkershh,share_secondary,...,"sector1d_Other community, social and personal service activities",sector1d_Public administration and defence,"sector1d_Real estate, renting and business activities","sector1d_Transport, storage and communications",sector1d_Wholesale and retail trade,sector1d_nan,utl_exp_ppp17,weight,survey_id,hhid
0,4,0,1,0,50,2,1,0,0.666667,0.333333,...,1,0,0,0,0,0,567.80914,320,400000,400001
1,6,0,1,1,65,3,1,0,0.500000,0.250000,...,0,0,0,0,0,1,561.70367,480,400000,400002
2,4,0,0,0,66,0,4,0,1.000000,0.000000,...,0,0,0,0,0,0,183.16423,320,400000,400003
3,4,0,0,1,50,1,2,0,0.333333,0.000000,...,0,0,0,0,0,1,696.02411,320,400000,400004
4,4,0,0,1,63,2,1,0,1.000000,0.333333,...,0,0,0,0,0,0,286.95731,320,400000,400005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103018,3,0,0,0,63,2,1,0,1.000000,0.333333,...,0,0,0,0,0,0,402.71024,963,600000,634209
103019,7,1,2,2,42,1,1,0,1.000000,1.000000,...,0,0,0,0,0,0,238.44685,2247,600000,634210
103020,3,0,0,1,62,1,1,0,1.000000,0.000000,...,0,0,0,0,0,0,211.95276,963,600000,634211
103021,2,0,0,0,67,1,0,1,1.000000,0.500000,...,0,0,0,0,0,0,254.34331,642,600000,634212


In [8]:
survey_id = test_features['survey_id']
hhid =  test_features['hhid']

In [9]:
columns_to_drop = ['weight','survey_id','hhid']
input_data = test_features.drop(columns=columns_to_drop)
input_data

,hsize,num_children5,num_children10,num_children18,age,num_adult_female,num_adult_male,num_elderly,sworkershh,share_secondary,...,sector1d_Hotels and restaurants,sector1d_Manufacturing,sector1d_Mining and quarrying,"sector1d_Other community, social and personal service activities",sector1d_Public administration and defence,"sector1d_Real estate, renting and business activities","sector1d_Transport, storage and communications",sector1d_Wholesale and retail trade,sector1d_nan,utl_exp_ppp17
0,4,0,1,0,50,2,1,0,0.666667,0.333333,...,0,0,0,1,0,0,0,0,0,567.80914
1,6,0,1,1,65,3,1,0,0.500000,0.250000,...,0,0,0,0,0,0,0,0,1,561.70367
2,4,0,0,0,66,0,4,0,1.000000,0.000000,...,0,0,0,0,0,0,0,0,0,183.16423
3,4,0,0,1,50,1,2,0,0.333333,0.000000,...,0,0,0,0,0,0,0,0,1,696.02411
4,4,0,0,1,63,2,1,0,1.000000,0.333333,...,0,1,0,0,0,0,0,0,0,286.95731
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103018,3,0,0,0,63,2,1,0,1.000000,0.333333,...,0,0,0,0,0,0,0,0,0,402.71024
103019,7,1,2,2,42,1,1,0,1.000000,1.000000,...,0,0,0,0,0,0,0,0,0,238.44685
103020,3,0,0,1,62,1,1,0,1.000000,0.000000,...,0,0,0,0,0,0,0,0,0,211.95276
103021,2,0,0,0,67,1,0,1,1.000000,0.500000,...,0,0,0,0,0,0,0,0,0,254.34331


In [10]:
input_data.columns

Index(['hsize', 'num_children5', 'num_children10', 'num_children18', 'age',
       'num_adult_female', 'num_adult_male', 'num_elderly', 'sworkershh',
       'share_secondary', 'sfworkershh', 'region1', 'region2', 'region3',
       'region4', 'region5', 'region6', 'region7', 'dependency_ratio',
       'worker_share', 'log_hsize', 'educ_max_numeric', 'urban_numeric',
       'urban_educ_interaction', 'housing_index_score', 'water_source_numeric',
       'sanitation_source_numeric', 'employed_numeric', 'nonagric_numeric',
       'male_Female', 'male_Male', 'male_nan', 'owner_Not owner',
       'owner_Owner', 'owner_nan', 'water_Access', 'water_No access',
       'water_nan', 'toilet_Access', 'toilet_No access', 'toilet_nan',
       'sewer_Access', 'sewer_No access', 'sewer_nan', 'elect_Access',
       'elect_No access', 'elect_nan', 'dweltyp_Detached house',
       'dweltyp_Improvised housing unit', 'dweltyp_Other',
       'dweltyp_Separate apartment', 'dweltyp_Several buildings connected'

In [11]:
train_columns = ['hsize', 'num_children5', 'num_children10', 'num_children18', 'age',
       'num_adult_female', 'num_adult_male', 'num_elderly', 'sworkershh',
       'share_secondary', 'sfworkershh', 'region1', 'region2', 'region3',
       'region4', 'region5', 'region6', 'region7', 'dependency_ratio',
       'worker_share', 'log_hsize', 'educ_max_numeric', 'urban_numeric',
       'urban_educ_interaction', 'housing_index_score', 'water_source_numeric',
       'sanitation_source_numeric', 'employed_numeric', 'nonagric_numeric',
       'male_Female', 'male_Male', 'male_nan', 'owner_Not owner',
       'owner_Owner', 'owner_nan', 'water_Access', 'water_No access',
       'water_nan', 'toilet_Access', 'toilet_No access', 'toilet_nan',
       'sewer_Access', 'sewer_No access', 'sewer_nan', 'elect_Access',
       'elect_No access', 'elect_nan', 'dweltyp_Detached house',
       'dweltyp_Improvised housing unit', 'dweltyp_Other',
       'dweltyp_Separate apartment', 'dweltyp_Several buildings connected',
       'dweltyp_nan',
       'sector1d_Activities of private households as employers ',
       'sector1d_Agriculture, hunting and forestry', 'sector1d_Construction',
       'sector1d_Education', 'sector1d_Electricity, gas and water supply',
       'sector1d_Financial intermediation', 'sector1d_Fishing',
       'sector1d_Health and social work', 'sector1d_Hotels and restaurants',
       'sector1d_Manufacturing', 'sector1d_Mining and quarrying',
       'sector1d_Other community, social and personal service activities',
       'sector1d_Public administration and defence',
       'sector1d_Real estate, renting and business activities',
       'sector1d_Transport, storage and communications',
       'sector1d_Wholesale and retail trade', 'sector1d_nan', 'utl_exp_ppp17']

In [12]:
x_test_aligned = input_data.reindex(columns=train_columns, fill_value=0)#fix any mismatch in the columns
x_test_aligned

,hsize,num_children5,num_children10,num_children18,age,num_adult_female,num_adult_male,num_elderly,sworkershh,share_secondary,...,sector1d_Hotels and restaurants,sector1d_Manufacturing,sector1d_Mining and quarrying,"sector1d_Other community, social and personal service activities",sector1d_Public administration and defence,"sector1d_Real estate, renting and business activities","sector1d_Transport, storage and communications",sector1d_Wholesale and retail trade,sector1d_nan,utl_exp_ppp17
0,4,0,1,0,50,2,1,0,0.666667,0.333333,...,0,0,0,1,0,0,0,0,0,567.80914
1,6,0,1,1,65,3,1,0,0.500000,0.250000,...,0,0,0,0,0,0,0,0,1,561.70367
2,4,0,0,0,66,0,4,0,1.000000,0.000000,...,0,0,0,0,0,0,0,0,0,183.16423
3,4,0,0,1,50,1,2,0,0.333333,0.000000,...,0,0,0,0,0,0,0,0,1,696.02411
4,4,0,0,1,63,2,1,0,1.000000,0.333333,...,0,1,0,0,0,0,0,0,0,286.95731
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103018,3,0,0,0,63,2,1,0,1.000000,0.333333,...,0,0,0,0,0,0,0,0,0,402.71024
103019,7,1,2,2,42,1,1,0,1.000000,1.000000,...,0,0,0,0,0,0,0,0,0,238.44685
103020,3,0,0,1,62,1,1,0,1.000000,0.000000,...,0,0,0,0,0,0,0,0,0,211.95276
103021,2,0,0,0,67,1,0,1,1.000000,0.500000,...,0,0,0,0,0,0,0,0,0,254.34331


In [13]:
x_test_aligned.columns == train_columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True])

In [14]:
scaler = StandardScaler()
x_test_scaled = scaler.fit_transform(x_test_aligned)
x_test_scaled.shape

(103023, 71)

In [15]:
y_pred_dl = saved_model.predict(x_test_scaled).flatten()

In [16]:
y_pred_dl

array([-0.2850451 , -0.56464857, -0.5769752 , ..., -0.593827  ,
       -0.5925904 , -0.73112553], shape=(103023,), dtype=float32)

In [17]:
target_scaler = joblib.load('target_scaler.pkl')
y_pred_dl_original = target_scaler.inverse_transform(y_pred_dl.reshape(-1, 1))
y_pred_dl_original

array([[8.705242 ],
       [5.9143333],
       [5.791293 ],
       ...,
       [5.623084 ],
       [5.6354275],
       [4.252616 ]], shape=(103023, 1), dtype=float32)

In [18]:
df = pd.DataFrame({
      'survey_id': survey_id.values.flatten(),
    'hhid': hhid.values.flatten(),
    'cons_ppp17': y_pred_dl_original.flatten(),
    'weight': test_features['weight'].values.flatten()
})
df

,survey_id,hhid,cons_ppp17,weight
0,400000,400001,8.705242,320
1,400000,400002,5.914333,480
2,400000,400003,5.791293,320
3,400000,400004,11.857430,320
4,400000,400005,4.793376,320
...,...,...,...,...
103018,600000,634209,4.906608,963
103019,600000,634210,2.944900,2247
103020,600000,634211,5.623084,963
103021,600000,634212,5.635427,642


In [19]:
df_400000 = df[df['survey_id'] == 400000].copy()
df_500000 = df[df['survey_id'] == 500000].copy()
df_600000 = df[df['survey_id'] == 600000].copy()

In [20]:
df_600000

,survey_id,hhid,cons_ppp17,weight
68810,600000,600001,16.220795,45
68811,600000,600002,9.678142,45
68812,600000,600003,17.167086,135
68813,600000,600004,2.349926,135
68814,600000,600005,8.229864,672
...,...,...,...,...
103018,600000,634209,4.906608,963
103019,600000,634210,2.944900,2247
103020,600000,634211,5.623084,963
103021,600000,634212,5.635427,642


In [21]:
def get_poverty_rates(predictions, weights, thresholds):
    predictions = predictions.ravel() 
    weights = weights.ravel()
    total_population = np.sum(weights)
    
    rates = {}
    for t in thresholds:
        is_poor = predictions < t
        poor_population = np.sum(weights[is_poor])
        rate = poor_population / total_population
        rates[f'pct_hh_below_{t}'] = rate
        
    return rates

In [22]:
thresholds=np.array([ 3.17,  3.94,  4.6 ,  5.26,  5.88,  6.47,  7.06,  7.7 ,  8.4 ,
        9.13,  9.87, 10.7 , 11.62, 12.69, 14.03, 15.64, 17.76, 20.99,
       27.37])

In [23]:
predictions = df_400000['cons_ppp17'].values
weights = df_400000['weight'].values
rates_40000 = get_poverty_rates(predictions, weights, thresholds)
#rates_40000

In [24]:
predictions = df_500000['cons_ppp17'].values
weights = df_500000['weight'].values
rates_50000 = get_poverty_rates(predictions, weights, thresholds)
#rates_50000

In [25]:
predictions = df_600000['cons_ppp17'].values
weights = df_600000['weight'].values
rates_60000 = get_poverty_rates(predictions, weights, thresholds)
#rates_60000

In [26]:
consumption_template = pd.read_csv('data/submission_formal_template/predicted_household_consumption.csv')#see the submission format
consumption_template

,survey_id,hhid,cons_ppp17
0,400000,400001,0.0
1,400000,400002,0.0
2,400000,400003,0.0
3,400000,400004,0.0
4,400000,400005,0.0
...,...,...,...
103018,600000,634209,0.0
103019,600000,634210,0.0
103020,600000,634211,0.0
103021,600000,634212,0.0


In [27]:
pr_template = pd.read_csv('data/submission_formal_template/predicted_poverty_distribution.csv')#see the submission format
pr_template

,survey_id,pct_hh_below_3.17,pct_hh_below_3.94,pct_hh_below_4.60,pct_hh_below_5.26,pct_hh_below_5.88,pct_hh_below_6.47,pct_hh_below_7.06,pct_hh_below_7.70,pct_hh_below_8.40,pct_hh_below_9.13,pct_hh_below_9.87,pct_hh_below_10.70,pct_hh_below_11.62,pct_hh_below_12.69,pct_hh_below_14.03,pct_hh_below_15.64,pct_hh_below_17.76,pct_hh_below_20.99,pct_hh_below_27.37
0,400000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,500000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,600000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
pr_template.columns

Index(['survey_id', 'pct_hh_below_3.17', 'pct_hh_below_3.94',
       'pct_hh_below_4.60', 'pct_hh_below_5.26', 'pct_hh_below_5.88',
       'pct_hh_below_6.47', 'pct_hh_below_7.06', 'pct_hh_below_7.70',
       'pct_hh_below_8.40', 'pct_hh_below_9.13', 'pct_hh_below_9.87',
       'pct_hh_below_10.70', 'pct_hh_below_11.62', 'pct_hh_below_12.69',
       'pct_hh_below_14.03', 'pct_hh_below_15.64', 'pct_hh_below_17.76',
       'pct_hh_below_20.99', 'pct_hh_below_27.37'],
      dtype='object')

In [29]:
desired_columns = ['survey_id', 'pct_hh_below_3.17', 'pct_hh_below_3.94',
       'pct_hh_below_4.60', 'pct_hh_below_5.26', 'pct_hh_below_5.88',
       'pct_hh_below_6.47', 'pct_hh_below_7.06', 'pct_hh_below_7.70',
       'pct_hh_below_8.40', 'pct_hh_below_9.13', 'pct_hh_below_9.87',
       'pct_hh_below_10.70', 'pct_hh_below_11.62', 'pct_hh_below_12.69',
       'pct_hh_below_14.03', 'pct_hh_below_15.64', 'pct_hh_below_17.76',
       'pct_hh_below_20.99', 'pct_hh_below_27.37']

row_400 = rates_40000.copy()
row_400['survey_id'] = 400000

row_500 = rates_50000.copy()
row_500['survey_id'] = 500000

row_600 = rates_60000.copy()
row_600['survey_id'] = 600000

df_final = pd.DataFrame([row_400, row_500, row_600])
df_final.columns = df_final.columns.str.replace(r'4.6$', '4.60', regex=True)
df_final.columns = df_final.columns.str.replace(r'7.7$', '7.70', regex=True)
df_final.columns = df_final.columns.str.replace(r'8.4$', '8.40', regex=True)
df_final.columns = df_final.columns.str.replace(r'10.7$', '10.70', regex=True)
df_final = df_final[desired_columns]
df_final_pr = df_final.copy()
df_final_pr

,survey_id,pct_hh_below_3.17,pct_hh_below_3.94,pct_hh_below_4.60,pct_hh_below_5.26,pct_hh_below_5.88,pct_hh_below_6.47,pct_hh_below_7.06,pct_hh_below_7.70,pct_hh_below_8.40,pct_hh_below_9.13,pct_hh_below_9.87,pct_hh_below_10.70,pct_hh_below_11.62,pct_hh_below_12.69,pct_hh_below_14.03,pct_hh_below_15.64,pct_hh_below_17.76,pct_hh_below_20.99,pct_hh_below_27.37
0,400000,0.031478,0.089151,0.157583,0.229167,0.299200,0.365800,0.435091,0.506578,0.578599,0.644167,0.701306,0.753230,0.799438,0.844980,0.888403,0.921253,0.950284,0.972195,0.990427
1,500000,0.022996,0.075622,0.136617,0.208560,0.281245,0.349411,0.418036,0.489999,0.567243,0.634744,0.695248,0.747799,0.794928,0.836659,0.877481,0.915581,0.947047,0.971110,0.990352
2,600000,0.023620,0.072263,0.131606,0.195912,0.257473,0.324138,0.391846,0.470733,0.544613,0.617442,0.680333,0.737114,0.789420,0.835948,0.877580,0.914301,0.946685,0.970788,0.990285


In [30]:
df_final_cons = df.drop(columns=['weight'])
df_final_cons

,survey_id,hhid,cons_ppp17
0,400000,400001,8.705242
1,400000,400002,5.914333
2,400000,400003,5.791293
3,400000,400004,11.857430
4,400000,400005,4.793376
...,...,...,...
103018,600000,634209,4.906608
103019,600000,634210,2.944900
103020,600000,634211,5.623084
103021,600000,634212,5.635427


In [31]:
if save:
    df_final_pr.to_csv(pr_file_name, index=False)
    df_final_cons.to_csv(cons_file_name, index=False)